# 計算每個文檔的token數量

In [1]:
import os
import json
from transformers import AutoTokenizer
from tqdm import tqdm

# ===== 設定多個資料夾路徑 =====
base_folders = [
    "../../coliee_dataset/task1/processed",
    "../../coliee_dataset/task1/processed_new",
    "../../coliee_dataset/task1/summary"
]

# 載入 tokenizer
tokenizer = AutoTokenizer.from_pretrained("CSHaitao/SAILER_en_finetune")

for dataset_path in base_folders:
    output_json = os.path.join(dataset_path, "document_token_stats.json")

    # 讀取文檔
    doc_token_info = []

    print(f"Reading and tokenizing documents in {dataset_path}...")
    for filename in tqdm(os.listdir(dataset_path)):
        if filename.endswith(".txt"):
            doc_id = filename.replace(".txt", "")
            file_path = os.path.join(dataset_path, filename)

            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read().strip()

            token_count = len(tokenizer.encode(text, truncation=False))
            doc_token_info.append({
                "id": doc_id,
                "token_count": token_count
            })

    # 儲存 JSON
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(doc_token_info, f, indent=2, ensure_ascii=False)

    print(f"已儲存 token 統計資訊到 {output_json}")


/home/lab/miniconda3/envs/THUIR-COLIEE2023-WSL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading and tokenizing documents in ../../coliee_dataset/task1/processed...


100%|██████████| 7353/7353 [00:41<00:00, 178.49it/s]


已儲存 token 統計資訊到 ../../coliee_dataset/task1/processed/document_token_stats.json
Reading and tokenizing documents in ../../coliee_dataset/task1/processed_new...


100%|██████████| 7353/7353 [00:10<00:00, 703.86it/s]


已儲存 token 統計資訊到 ../../coliee_dataset/task1/processed_new/document_token_stats.json
Reading and tokenizing documents in ../../coliee_dataset/task1/summary...


100%|██████████| 969/969 [00:00<00:00, 3535.15it/s]

已儲存 token 統計資訊到 ../../coliee_dataset/task1/summary/document_token_stats.json


## 畫統計圖

In [2]:
import os
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# ===== 可調參數 =====
bin_size = 513
max_token_display = 10000

# ===== 資料夾清單 =====
base_folders = [
    "../../coliee_dataset/task1/processed",
    "../../coliee_dataset/task1/processed_new",
    "../../coliee_dataset/task1/summary"
]

for folder in base_folders:
    input_json = os.path.join(folder, "document_token_stats.json")
    output_plot = os.path.join(folder, "token_distribution_from_json.png")

    with open(input_json, 'r', encoding='utf-8') as f:
        doc_token_info = json.load(f)

    token_counts = [item["token_count"] for item in doc_token_info]

    # 計算平均
    average_tokens = sum(token_counts) / len(token_counts)
    print(f"[{folder}] Average token count: {average_tokens:.2f}")

    # 過濾顯示範圍內的 token 數
    filtered_token_counts = [t for t in token_counts if t <= max_token_display]

    # 分 bin 畫圖
    bins = list(range(0, (max_token_display // bin_size + 1) * bin_size + 1, bin_size))
    hist, bin_edges = np.histogram(filtered_token_counts, bins=bins)

    # 畫圖
    plt.figure(figsize=(10, 6))
    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])-1}" for i in range(len(hist))]
    plt.bar(labels, hist)
    plt.xticks(rotation=45)
    plt.xlabel("Token Count Range (≤ {} tokens)".format(max_token_display))
    plt.ylabel("Number of Documents")
    plt.title("Document Length (Token Count) Distribution")
    plt.tight_layout()
    plt.savefig(output_plot)
    print(f"圖表儲存為 {output_plot}")
    plt.close()

[../../coliee_dataset/task1/processed] Average token count: 4980.33
圖表儲存為 ../../coliee_dataset/task1/processed/token_distribution_from_json.png
[../../coliee_dataset/task1/processed_new] Average token count: 1162.08
圖表儲存為 ../../coliee_dataset/task1/processed_new/token_distribution_from_json.png
[../../coliee_dataset/task1/summary] Average token count: 125.18
圖表儲存為 ../../coliee_dataset/task1/summary/token_distribution_from_json.png


# 計算訓練及驗證資料分別查詢數量、每個查詢平均相關數

In [3]:
# 訓練資料有幾個query
from typing import List
import pandas as pd

def load_query_ids(path: str) -> List[str]:
    with open(path, 'r') as f:
        return [line.strip() for line in f if line.strip()]
    
base_folder = "../../coliee_dataset/task1"

# 訓練資料query數量
train_qid_path = base_folder+'/train_qid.tsv'
train_qids = load_query_ids(train_qid_path)
display("訓練資料query數量: "+str(len(train_qids)))

# 驗證資料query數量
valid_qid_path = base_folder+'/valid_qid.tsv'
valid_qids = load_query_ids(valid_qid_path)
display("驗證資料query數量: "+str(len(valid_qids)))

# 讀訓練及驗證的標註資料
import json
with open(base_folder+'/task1_train_labels_2025.json', 'r', encoding='utf-8') as f:
    train_labels_json = json.load(f)
# print(type(train_labels_json))

# ---訓練資料相關數量分析---
# 儲存所有query相關數量
train_query_relevance_counts = {}
for qid in train_qids:
    qid_txt = qid + '.txt'
    if qid_txt not in train_labels_json:
        print(f"警告: query id {qid} 不在 task1_train_labels_2025.json 中")
    else:
        train_query_relevance_counts[qid] = len(train_labels_json[qid_txt])
        # print(f"query id {qid} 的相關數量: {len(train_labels_json[qid_txt])}")
# 計算平均相關數量
if train_query_relevance_counts:
    avg_relevance = sum(train_query_relevance_counts.values()) / len(train_query_relevance_counts)
    print(f"訓練資料query的平均相關數量: {avg_relevance}")
else:
    print("訓練資料沒有有效的query相關數量")
# 畫出相關數量分佈圖(互動式)
import plotly.express as px
hist_data = list(train_query_relevance_counts.values())
fig = px.histogram(
    x=hist_data,
    title="訓練資料query的相關數量分佈",
    labels={"x": "相關數量", "y": "查詢數量"}  # 這裡改掉 hover 的名稱
)
fig.update_xaxes(title="相關數量")
fig.update_yaxes(title="查詢數量")
fig.show()


# ---驗證資料相關數量分析---
# 儲存所有query相關數量
valid_query_relevance_counts = {}
for qid in valid_qids:
    qid_txt = qid + '.txt'
    if qid_txt not in train_labels_json:
        print(f"警告: query id {qid} 不在 task1_train_labels_2025.json 中")
    else:
        valid_query_relevance_counts[qid] = len(train_labels_json[qid_txt])
        # print(f"query id {qid} 的相關數量: {len(train_labels_json[qid_txt])}")
# 計算平均相關數量
if valid_query_relevance_counts:
    avg_relevance = sum(valid_query_relevance_counts.values()) / len(valid_query_relevance_counts)
    print(f"驗證資料query的平均相關數量: {avg_relevance}")
else:
    print("驗證資料沒有有效的query相關數量")
hist_data = list(valid_query_relevance_counts.values())
fig = px.histogram(
    x=hist_data,
    title="驗證資料query的相關數量分佈",
    labels={"x": "相關數量", "y": "查詢數量"}  # 這裡改掉 hover 的名稱
)
fig.update_xaxes(title="相關數量")
fig.update_yaxes(title="查詢數量")
fig.show()

'訓練資料query數量: 1341'

'驗證資料query數量: 336'

訓練資料query的平均相關數量: 4.067114093959732


驗證資料query的平均相關數量: 4.241071428571429
